## Evaluate Presidio Analyzer using the Presidio Evaluator framework

In [49]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator

In [50]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2

In [51]:
dataset_name = "data_person_10_zh_presidio_format.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name), token_model_version="zh_core_web_sm")
print(len(dataset))
print(dataset[0].full_text if dataset else "dataset is empty")

tokenizing input: 100%|██████████| 1/1 [00:00<00:00, 13.38it/s]

1
柯晓梅，51岁女性，现居吉林省长春市朝阳区延安大街74号，职业为水管工。她因出现呼吸急促、咳嗽和胸闷等症状被诊断为哮喘，目前正在温迪·格伦医生的指导下服用布洛芬进行治疗。柯晓梅的个人信息包括身份证号220105197307013827，电子邮箱kexiaomei@163.com，手机号码15943218765。她的信用评分为756分，年收入为1,170,289.42元。相关医疗或财务记录的交易明细中包含标识符"INDO GIBL Indiaforensic STL19071"。


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [52]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        print("SAMPLE:", sample)
        print("NUM_TAGS:", len(sample.tags))
        for tag in sample.tags:
            print("TAG:", tag)
            entity_counter[tag] += 1
    return entity_counter

In [53]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

print("\nExample InputSample:")
print(dataset[0])

SAMPLE: Full text: 柯晓梅，51岁女性，现居吉林省长春市朝阳区延安大街74号，职业为水管工。她因出现呼吸急促、咳嗽和胸闷等症状被诊断为哮喘，目前正在温迪·格伦医生的指导下服用布洛芬进行治疗。柯晓梅的个人信息包括身份证号220105197307013827，电子邮箱kexiaomei@163.com，手机号码15943218765。她的信用评分为756分，年收入为1,170,289.42元。相关医疗或财务记录的交易明细中包含标识符"INDO GIBL Indiaforensic STL19071"。
Spans: [Span(type: PERSON, value: 柯晓梅, char_span: [0: 3]), Span(type: AGE, value: 51, char_span: [4: 6]), Span(type: GENDER, value: 女, char_span: [7: 8]), Span(type: LOCATION, value: 吉林省长春市朝阳区延安大街74号, char_span: [12: 28]), Span(type: PROFESSION, value: 水管工, char_span: [32: 35]), Span(type: DIAGNOSIS, value: 哮喘, char_span: [57: 59]), Span(type: PERSON, value: 温迪·格伦, char_span: [64: 69]), Span(type: MEDICATION, value: 布洛芬, char_span: [77: 80]), Span(type: PERSON, value: 柯晓梅, char_span: [85: 88]), Span(type: ID, value: 220105197307013827, char_span: [99: 117]), Span(type: EMAIL_ADDRESS, value: kexiaomei@163.com, char_span: [122: 139]), Span(type: PHONE_NUMBER, value: 15943218765, char_span: [144: 155]), Span(type: CREDIT_SCORE, valu

In [54]:
print("A few examples sentences containing each entity:\n")
for entity in entity_counts.keys():
    samples = [sample for sample in dataset if entity in set(sample.tags)]
    if len(samples) > 1 and entity != "O":
        print(f"Entity: <{entity}> two example sentences:\n"
              f"\n1) {samples[0].full_text}"
              f"\n2) {samples[1].full_text}"
              f"\n------------------------------------\n")

A few examples sentences containing each entity:



In [55]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# Loading the vanilla Analyzer Engine, with the default NER model.
configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "zh", "model_name": "zh_core_web_sm"}],
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine_with_sm = provider.create_engine()
analyzer_engine = AnalyzerEngine(nlp_engine=nlp_engine_with_sm, supported_languages=["zh"], default_score_threshold=0.4)

In [56]:
from presidio_evaluator.models import  PresidioAnalyzerWrapper

entities_mapping=PresidioAnalyzerWrapper.presidio_entities_map # default mapping
pprint(entities_mapping)

dataset = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)
new_entity_counts = get_entity_counts(dataset)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.values())


{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'PREFIX': 'TITLE',
 'SSN': 'US_SSN',
 'STAFF': 'PE

In [57]:
# Test Analyzer
text=dataset[0].full_text
res = analyzer_engine.analyze(text=text, 
                              language="zh", 
                              return_decision_process=True)
for result in res:
    print(f"\nEntity: {result.entity_type}, Text: {text[result.start:result.end]}\n\nAnalysis explanation:")
    pprint(result.analysis_explanation)


Entity: EMAIL_ADDRESS, Text: 电子邮箱kexiaomei@163.com

Analysis explanation:
{'recognizer': 'EmailRecognizer', 'pattern_name': 'Email (Medium)', 'pattern': "\\b((([!#$%&'*+\\-/=?^_`{|}~\\w])|([!#$%&'*+\\-/=?^_`{|}~\\w][!#$%&'*+\\-/=?^_`{|}~\\.\\w]{0,}[!#$%&'*+\\-/=?^_`{|}~\\w]))[@]\\w+([-.]\\w+)*\\.\\w+([-.]\\w+)*)\\b", 'original_score': 0.5, 'score': 1.0, 'textual_explanation': 'Detected by `EmailRecognizer` using pattern `Email (Medium)`', 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': True, 'regex_flags': 26}

Entity: PERSON, Text: 柯晓梅

Analysis explanation:
{'recognizer': 'SpacyRecognizer', 'pattern_name': None, 'pattern': None, 'original_score': 0.85, 'score': 0.85, 'textual_explanation': "Identified as PERSON by Spacy's Named Entity Recognition", 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': None}

Entity: DATE_TIME, Text: 51岁

Analysis explanation:
{'recognizer': 'SpacyRecognizer', 'pat

In [58]:
# 找到包含"51岁"的样本，打印其 token 与 tag 对应关系
for i, s in enumerate(dataset):
    if "51岁" in s.full_text:
        print(f"sample {i}: {s.full_text[:40]}")
        for tok, tag in zip(s.tokens, s.tags):
            if "51" in str(tok) or tag != "O":
                print(f"  tok={tok!r:15}  tag={tag}")
        print()

sample 0: 柯晓梅，51岁女性，现居吉林省长春市朝阳区延安大街74号，职业为水管工。她因出现
  tok=柯晓梅              tag=PERSON
  tok=51               tag=AGE
  tok=吉林省              tag=LOCATION
  tok=长春市              tag=LOCATION
  tok=朝阳区              tag=LOCATION
  tok=延安               tag=LOCATION
  tok=大街               tag=LOCATION
  tok=74号              tag=LOCATION
  tok=温迪·格伦            tag=PERSON
  tok=柯晓梅              tag=PERSON
  tok=220105197307013827  tag=ID
  tok=kexiaomei        tag=EMAIL_ADDRESS
  tok=@                tag=EMAIL_ADDRESS
  tok=163.com          tag=EMAIL_ADDRESS
  tok=15943218765      tag=PHONE_NUMBER



In [59]:
# Set up the experiment tracker to log the experiment for reproducibility
experiment = get_experiment_tracker()

# Create the evaluator object
evaluator = SpanEvaluator(model=analyzer_engine)


# Track model and dataset params
params = {"dataset_name": dataset_name, 
          "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

--------
Entities supported by this Presidio Analyzer instance:
DATE_TIME, MAC_ADDRESS, ID, EMAIL, ORGANIZATION, CRYPTO, NRP, LOCATION, IBAN_CODE, PHONE_NUMBER, AGE, MEDICAL_LICENSE, IP_ADDRESS, PERSON, EMAIL_ADDRESS, URL


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


## 6. Run experiment

In [60]:
%%time

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset)
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

Running model PresidioAnalyzerWrapper on dataset...
Finished running model on dataset
saving experiment data to /Users/jyu36/code/ic-llm/presidio/experiment_20260316-212421.json
CPU times: user 31.2 ms, sys: 8.18 ms, total: 39.4 ms
Wall time: 53.3 ms


In [61]:
pprint({"PII F":results.pii_f, "PII recall": results.pii_recall, "PII precision": results.pii_precision})

{'PII F': 0.5, 'PII precision': 0.5, 'PII recall': 0.5}


### 7a. False positives
#### Most common false positive tokens:

In [62]:
results.model_errors

[<ModelError type: ErrorType.FN, Annotation = O, prediction = AGE, Token = 51, Full text = 51, Metadata = None,
 <ModelError type: ErrorType.FP, Annotation = DATE_TIME, prediction = O, Token = 51 岁, Full text = 51 岁, Metadata = None,
 <ModelError type: ErrorType.FN, Annotation = O, prediction = LOCATION, Token = 吉林省 长春市 朝阳区 延安 大街 74号, Full text = 吉林省 长春市 朝阳区 延安 大街 74号, Metadata = None,
 <ModelError type: ErrorType.FP, Annotation = LOCATION, prediction = O, Token = 吉林省, Full text = 吉林省, Metadata = None,
 <ModelError type: ErrorType.FP, Annotation = DATE_TIME, prediction = O, Token = 74号, Full text = 74号, Metadata = None,
 <ModelError type: ErrorType.FN, Annotation = O, prediction = ID, Token = 220105197307013827, Full text = 220105197307013827, Metadata = None,
 <ModelError type: ErrorType.FN, Annotation = O, prediction = EMAIL_ADDRESS, Token = kexiaomei 163.com, Full text = kexiaomei @ 163.com, Metadata = None,
 <ModelError type: ErrorType.FP, Annotation = EMAIL_ADDRESS, prediction = O

In [63]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('布洛芬', 2),
 ('51 岁', 1),
 ('吉林省', 1),
 ('74号', 1),
 ('电子 邮箱 kexiaomei 163.com', 1)]
---------------
Example sentence with each FP token:
	- 布洛芬 (`布洛芬` pred as PERSON)
	- 51 岁 (`51 岁` pred as O)
	- 吉林省 (`吉林省` pred as O)
	- 74号 (`74号` pred as O)
	- 电子 邮箱 kexiaomei @ 163.com (`电子 邮箱 kexiaomei 163.com` pred as O)


[('布洛芬', 2),
 ('51 岁', 1),
 ('吉林省', 1),
 ('74号', 1),
 ('电子 邮箱 kexiaomei 163.com', 1)]

#### More FP analysis

In [64]:
fps_df = ModelError.get_fps_dataframe(results.model_errors)
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,51 岁,51 岁,DATE_TIME,O
1,吉林省,吉林省,LOCATION,O
2,74号,74号,DATE_TIME,O
3,电子 邮箱 kexiaomei @ 163.com,电子 邮箱 kexiaomei 163.com,EMAIL_ADDRESS,O
4,布洛芬,布洛芬,O,PERSON
5,布洛芬,布洛芬,O,PII


### 7b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [65]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('51', 1),
 ('吉林省 长春市 朝阳区 延安 大街 74号', 1),
 ('220105197307013827', 1),
 ('kexiaomei 163.com', 1)]
---------------
Example sentence with each FN token:
	- 51 (`51` annotated as O)
	- 吉林省 长春市 朝阳区 延安 大街 74号 (`吉林省 长春市 朝阳区 延安 大街 74号` annotated as O)
	- 220105197307013827 (`220105197307013827` annotated as O)
	- kexiaomei @ 163.com (`kexiaomei 163.com` annotated as O)


[('51', 1),
 ('吉林省 长春市 朝阳区 延安 大街 74号', 1),
 ('220105197307013827', 1),
 ('kexiaomei 163.com', 1)]

#### More FN analysis

In [66]:
fns_df = ModelError.get_fns_dataframe(results.model_errors)

In [67]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,51,51,O,AGE
1,吉林省 长春市 朝阳区 延安 大街 74号,吉林省 长春市 朝阳区 延安 大街 74号,O,LOCATION
2,220105197307013827,220105197307013827,O,ID
3,kexiaomei @ 163.com,kexiaomei 163.com,O,EMAIL_ADDRESS


In [68]:
# 事件级视图（补加版）：一个 token 事件只保留一条记录
# 将 WrongEntity 及其派生的 O->X / Y->O 记录合并

import pandas as pd


def _err_to_row(e, idx):
    return {
        "idx": idx,
        "error_type": str(getattr(e, "error_type", None)),
        "annotation": str(getattr(e, "annotation", None)),
        "prediction": str(getattr(e, "prediction", None)),
        "token": getattr(e, "token", None),
        "full_text": getattr(e, "full_text", None),
        "metadata": getattr(e, "metadata", None),
        "explanation": getattr(e, "explanation", None),
        "_used": False,
    }


def collapse_model_errors_to_events(model_errors, token_filter=None):
    rows = [_err_to_row(e, i) for i, e in enumerate(model_errors)]
    if token_filter is not None:
        rows = [r for r in rows if r["token"] == token_filter]

    wrong = [r for r in rows if r["error_type"].endswith("WrongEntity")]
    fn_like = [
        r
        for r in rows
        if r["error_type"].endswith("FN") and r["annotation"] == "O" and r["prediction"] != "O"
    ]
    fp_like = [
        r
        for r in rows
        if r["error_type"].endswith("FP") and r["annotation"] != "O" and r["prediction"] == "O"
    ]

    def same_context(a, b):
        return (
            a["token"] == b["token"]
            and a["full_text"] == b["full_text"]
            and str(a["metadata"]) == str(b["metadata"])
        )

    collapsed = []

    for w in wrong:
        if w["_used"]:
            continue

        ann = w["annotation"]
        pred = w["prediction"]

        match_fn = None
        for f in fn_like:
            if not f["_used"] and same_context(w, f) and f["prediction"] == pred:
                match_fn = f
                break

        match_fp = None
        for f in fp_like:
            if not f["_used"] and same_context(w, f) and f["annotation"] == ann:
                match_fp = f
                break

        w["_used"] = True
        if match_fn:
            match_fn["_used"] = True
        if match_fp:
            match_fp["_used"] = True

        collapsed.append(
            {
                "event_type": "WrongEntity",
                "annotation": ann,
                "prediction": pred,
                "token": w["token"],
                "explanation": w["explanation"],
                "source_idxs": [x for x in [w["idx"], match_fn["idx"] if match_fn else None, match_fp["idx"] if match_fp else None] if x is not None],
            }
        )

    for r in rows:
        if r["_used"]:
            continue
        collapsed.append(
            {
                "event_type": r["error_type"].split(".")[-1],
                "annotation": r["annotation"],
                "prediction": r["prediction"],
                "token": r["token"],
                "explanation": r["explanation"],
                "source_idxs": [r["idx"]],
            }
        )

    return pd.DataFrame(collapsed)


# 全量事件级视图
events_df = collapse_model_errors_to_events(results.model_errors)
print(f"raw model_errors: {len(results.model_errors)}")
print(f"collapsed events: {len(events_df)}")
display(events_df.head(50))

raw model_errors: 10
collapsed events: 10


,event_type,annotation,prediction,token,explanation,source_idxs
0,FN,O,AGE,51,Entity AGE not detected. iou with DATE_TIME=0.67 compared to threshold=0.9,[0]
1,FP,DATE_TIME,O,51 岁,"Entity DATE_TIME falsely detected, iou=0.67 compared to threshold=0.9",[1]
2,FN,O,LOCATION,吉林省 长春市 朝阳区 延安 大街 74号,Entity LOCATION not detected due to low iou=0.24 compared to threshold=0.9,[2]
3,FP,LOCATION,O,吉林省,"Entity LOCATION falsely detected, iou=0.24 compared to threshold=0.9",[3]
4,FP,DATE_TIME,O,74号,"Entity DATE_TIME falsely detected, iou=0.24 compared to threshold=0.9",[4]
5,FN,O,ID,220105197307013827,Entity ID not detected.,[5]
6,FN,O,EMAIL_ADDRESS,kexiaomei 163.com,Entity EMAIL_ADDRESS not detected due to low iou=0.81 compared to threshold=0.9,[6]
7,FP,EMAIL_ADDRESS,O,电子 邮箱 kexiaomei 163.com,"Entity EMAIL_ADDRESS falsely detected, iou=0.81 compared to threshold=0.9",[7]
8,FP,O,PERSON,布洛芬,False prediction with no overlap: PERSON,[8]
9,FP,O,PII,布洛芬,False prediction with no overlap: PII,[9]


In [69]:
display(events_df[events_df['token'] == '布洛芬'].head(50))

,event_type,annotation,prediction,token,explanation,source_idxs
8,FP,O,PERSON,布洛芬,False prediction with no overlap: PERSON,[8]
9,FP,O,PII,布洛芬,False prediction with no overlap: PII,[9]
